## Imports & Configuration

In [1]:
import pandas as pd
import numpy as np

from IPython.display import display

# Reproducibility
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Source dataset (GitHub)
CSV_URL = (
    "https://raw.githubusercontent.com/semoglou/cart/main/"
    "llm_baselines/llm_taxonomies/"
    "CART_LLM_taxonomies_shared_paragraphs.csv"
)

## Load & Validate Dataset

In [2]:
# Load CSV directly from GitHub
df = pd.read_csv(CSV_URL, dtype={"paragraph_id": "string"})

# Required columns
required_columns = [
    "paragraph_id",
    "paragraph",
    "nf_cart",
    "nf_deepseek",
    "nf_llama"
]

# Check required columns
missing_columns = set(required_columns) - set(df.columns)

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

# Check for missing values
assert df[required_columns].notna().all().all(), (
    "Dataset contains missing values."
)

# Check for duplicate paragraph IDs
assert df["paragraph_id"].is_unique, (
    "Dataset contains duplicate paragraph IDs."
)

# Check for empty strings
for col in required_columns:
    assert df[col].astype(str).str.strip().ne("").all(), (
        f"Empty values found in {col}."
    )

# Check for Uncategorized narrative families
for col in ["nf_cart", "nf_deepseek", "nf_llama"]:
    assert not df[col].astype(str).str.strip().str.casefold().eq(
        "uncategorized"
    ).any(), (
        f"Uncategorized values found in {col}."
    )

# Display summary
print(f"Total paragraphs: {len(df):,}")
print(f"Unique paragraph IDs: {df['paragraph_id'].nunique():,}")

print("\nNarrative families per system:")
print(f"CART: {df['nf_cart'].nunique()}")
print(f"DeepSeek-R1: {df['nf_deepseek'].nunique()}")
print(f"Llama 3.3 70B: {df['nf_llama'].nunique()}")

display(df.head())

Total paragraphs: 2,617
Unique paragraph IDs: 2,617

Narrative families per system:
CART: 10
DeepSeek-R1: 18
Llama 3.3 70B: 24


,paragraph_id,paragraph,nf_cart,nf_deepseek,nf_llama
0,2023-03-07_1_10_p3,"Νομίζω ότι είναι για όλους μας, για τη χώρα μα...",Railway safety remains neglected and unsecured,Political Responsibility for Systemic Failures,Systemic Failures And Corruption
1,2023-03-07_1_10_p4,Προκύπτουν επομένως σοβαρά ερωτήματα για τις μ...,Systemic governance failures caused preventabl...,Systemic Governance Negligence Causation,Government Accountability Demanded
2,2023-03-07_1_18_p2,"Κύριε Υπουργέ, κυρίες και κύριοι συνάδελφοι, κ...",Government obstructs crash investigation to ev...,Institutional Corruption and Impunity,External Factors And Influence
3,2023-03-07_1_18_p4,"Κύριε Υπουργέ, συζητούμε ένα νομοσχέδιο σήμερα...",Negligence caused the preventable Tempi train ...,Independent Investigative Oversight Demands,Obstruction And Transparency Allegations
4,2023-03-07_1_18_p7,Ας έρθουμε να δούμε τι έκανε η Κυβέρνηση τα τε...,Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation,Safety Concerns And Regulations


## Create Blind Annotation Dataset

In [3]:
annotation_rows = []
mapping_rows = []

for annotation_id, (_, row) in enumerate(df.iterrows(), start=1):

    # Narrative families and their source systems
    options = [
        ("CART", row["nf_cart"]),
        ("DeepSeek-R1", row["nf_deepseek"]),
        ("Llama-3.3-70B", row["nf_llama"])
    ]

    # Randomize the order of A, B, C
    rng.shuffle(options)

    # Dataset visible to the judge
    annotation_rows.append({
        "annotation_id": annotation_id,
        "paragraph": row["paragraph"],
        "option_A": options[0][1],
        "option_B": options[1][1],
        "option_C": options[2][1],
        "option_D": "None of the above"
    })

    # Hidden mapping for subsequent analysis
    mapping_rows.append({
        "annotation_id": annotation_id,
        "paragraph_id": row["paragraph_id"],
        "system_A": options[0][0],
        "system_B": options[1][0],
        "system_C": options[2][0]
    })

# Create DataFrames
annotation_df = pd.DataFrame(annotation_rows)
mapping_df = pd.DataFrame(mapping_rows)

# Display results
print(f"Annotation rows: {len(annotation_df):,}")
print(f"Mapping rows: {len(mapping_df):,}")

print("\nBlind annotation dataset:")
display(annotation_df.head())

print("\nSystem mapping:")
display(mapping_df.head())

Annotation rows: 2,617
Mapping rows: 2,617

Blind annotation dataset:


,annotation_id,paragraph,option_A,option_B,option_C,option_D
0,1,"Νομίζω ότι είναι για όλους μας, για τη χώρα μα...",Systemic Failures And Corruption,Political Responsibility for Systemic Failures,Railway safety remains neglected and unsecured,None of the above
1,2,Προκύπτουν επομένως σοβαρά ερωτήματα για τις μ...,Systemic governance failures caused preventabl...,Government Accountability Demanded,Systemic Governance Negligence Causation,None of the above
2,3,"Κύριε Υπουργέ, κυρίες και κύριοι συνάδελφοι, κ...",Institutional Corruption and Impunity,External Factors And Influence,Government obstructs crash investigation to ev...,None of the above
3,4,"Κύριε Υπουργέ, συζητούμε ένα νομοσχέδιο σήμερα...",Obstruction And Transparency Allegations,Negligence caused the preventable Tempi train ...,Independent Investigative Oversight Demands,None of the above
4,5,Ας έρθουμε να δούμε τι έκανε η Κυβέρνηση τα τε...,Safety Concerns And Regulations,Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation,None of the above



System mapping:


,annotation_id,paragraph_id,system_A,system_B,system_C
0,1,2023-03-07_1_10_p3,Llama-3.3-70B,DeepSeek-R1,CART
1,2,2023-03-07_1_10_p4,CART,Llama-3.3-70B,DeepSeek-R1
2,3,2023-03-07_1_18_p2,DeepSeek-R1,Llama-3.3-70B,CART
3,4,2023-03-07_1_18_p4,Llama-3.3-70B,CART,DeepSeek-R1
4,5,2023-03-07_1_18_p7,Llama-3.3-70B,CART,DeepSeek-R1


## Validate Annotation Dataset

In [4]:
# 1. Check dataset sizes
assert len(annotation_df) == len(df) == 2617
assert len(mapping_df) == len(df)

# 2. Check unique annotation IDs
assert annotation_df["annotation_id"].is_unique
assert mapping_df["annotation_id"].is_unique

assert (
    annotation_df["annotation_id"].tolist()
    == mapping_df["annotation_id"].tolist()
)

# 3. Check missing values
assert annotation_df.notna().all().all()
assert mapping_df.notna().all().all()

# 4. Check None of the above
assert annotation_df["option_D"].eq("None of the above").all()

# 5. Check that each row contains all three systems
expected_systems = {
    "CART",
    "DeepSeek-R1",
    "Llama-3.3-70B"
}

for _, row in mapping_df.iterrows():
    actual_systems = {
        row["system_A"],
        row["system_B"],
        row["system_C"]
    }

    assert actual_systems == expected_systems

# 6. Verify that every randomized label matches its source
source_columns = {
    "CART": "nf_cart",
    "DeepSeek-R1": "nf_deepseek",
    "Llama-3.3-70B": "nf_llama"
}

for i in range(len(df)):
    for option in ["A", "B", "C"]:

        system = mapping_df.iloc[i][f"system_{option}"]
        actual_label = annotation_df.iloc[i][f"option_{option}"]

        expected_label = df.iloc[i][source_columns[system]]

        assert actual_label == expected_label, (
            f"Label mismatch at row {i}, option {option}"
        )

# 7. Display position distribution
print("System distribution across positions:\n")

for option in ["A", "B", "C"]:
    print(f"Position {option}:")
    print(mapping_df[f"system_{option}"].value_counts())
    print()

print("=" * 50)
print(f"Validated paragraphs: {len(annotation_df):,}")
print("=" * 50)

System distribution across positions:

Position A:
system_A
Llama-3.3-70B    906
CART             866
DeepSeek-R1      845
Name: count, dtype: int64

Position B:
system_B
CART             927
DeepSeek-R1      867
Llama-3.3-70B    823
Name: count, dtype: int64

Position C:
system_C
DeepSeek-R1      905
Llama-3.3-70B    888
CART             824
Name: count, dtype: int64

Validated paragraphs: 2,617


## Export CSV Files

In [5]:
from google.colab import files

# Output filenames
INPUT_FILENAME = "agentic_evaluation_input.csv"
MAPPING_FILENAME = "agentic_evaluation_mapping.csv"

# Export blind annotation dataset
annotation_df.to_csv(
    INPUT_FILENAME,
    index=False,
    encoding="utf-8-sig"
)

# Export hidden system mapping
mapping_df.to_csv(
    MAPPING_FILENAME,
    index=False,
    encoding="utf-8-sig"
)

# Verify exported files
saved_input = pd.read_csv(INPUT_FILENAME)
saved_mapping = pd.read_csv(MAPPING_FILENAME)

assert len(saved_input) == len(annotation_df)
assert len(saved_mapping) == len(mapping_df)

assert saved_input.columns.tolist() == annotation_df.columns.tolist()
assert saved_mapping.columns.tolist() == mapping_df.columns.tolist()

print(f"1. {INPUT_FILENAME}")
print(f"   Rows: {len(saved_input):,}")
print(f"   Columns: {saved_input.columns.tolist()}\n")

print(f"2. {MAPPING_FILENAME}")
print(f"   Rows: {len(saved_mapping):,}")
print(f"   Columns: {saved_mapping.columns.tolist()}")

# Download files
files.download(INPUT_FILENAME)
files.download(MAPPING_FILENAME)

1. agentic_evaluation_input.csv
   Rows: 2,617
   Columns: ['annotation_id', 'paragraph', 'option_A', 'option_B', 'option_C', 'option_D']

2. agentic_evaluation_mapping.csv
   Rows: 2,617
   Columns: ['annotation_id', 'paragraph_id', 'system_A', 'system_B', 'system_C']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>